# Bernstein–Vazirani: a secret in one query
**Quantum Computing · Week 5 (Quantum protocols and algorithms) · Day 4**

Lesson page: https://sara-aminpour.com/tutorials/qc/w5d4.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

**Bernstein–Vazirani:** a function $f(x) = s \cdot x \bmod 2$ hides a secret bit string $s$.
Classically you need one question per bit. With the Deutsch–Jozsa recipe, one quantum query
returns the whole of $s$.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler

secret = "100110"                                  # written like Qiskit prints: q5 ... q0

def f(x, s=secret):
    """s . x mod 2: the parity of the bits of x where s has a 1. Both are strings in Qiskit order."""
    return sum(int(a) * int(b) for a, b in zip(s, x)) % 2

# classical strategy: ask the inputs with a single 1; each answer is one bit of s
n = len(secret)
answers = []
for i in range(n):
    x = "".join("1" if k == i else "0" for k in range(n))
    answers.append(str(f(x)))
    print(f"ask x = {x} -> f(x) = {answers[-1]}")
print("recovered s =", "".join(answers), f"after {n} questions")

## One quantum query
The oracle needs no truth table: for every 1 in $s$, a CNOT from that input qubit to the output.
Mind the order: character $k$ of the string belongs to qubit $n - 1 - k$.

In [ ]:
def bv_circuit(s):
    n = len(s)
    qc = QuantumCircuit(n + 1, n)
    qc.x(n)
    qc.h(n)                                        # output qubit in |->
    qc.h(range(n))
    qc.barrier()
    for qubit, bit in enumerate(reversed(s)):      # qubit 0 is the LAST character
        if bit == "1":
            qc.cx(qubit, n)
    qc.barrier()
    qc.h(range(n))
    qc.measure(range(n), range(n))
    return qc

sampler = StatevectorSampler(seed=1)
counts = sampler.run([bv_circuit(secret)], shots=1000).result()[0].data.c.get_counts()
print("measured:", counts, "  secret:", secret)
bv_circuit(secret).draw("mpl", style="clifford")

## Why it works, one qubit at a time
Every input qubit stays unentangled the whole time. Each starts in $|0\rangle$, becomes $|+\rangle$,
and the oracle's CNOT kicks back a Z only onto the qubits where $s$ has a 1, turning them into
$|-\rangle$. The final H maps $|+\rangle \to |0\rangle$ and $|-\rangle \to |1\rangle$.

In [ ]:
from qiskit.quantum_info import Statevector, partial_trace

qc = bv_circuit(secret).remove_final_measurements(inplace=False)
state = Statevector(qc)
for qubit in range(n):
    others = [q for q in range(n + 1) if q != qubit]
    rho = partial_trace(state, others)
    p1 = rho.probabilities()[1]
    print(f"qubit {qubit}: P(1) = {p1:.3f}   purity = {np.real(np.trace(rho.data @ rho.data)):.3f}")
print("Every qubit is pure (purity 1) and certain: the answer is stored qubit by qubit.")

## Your turn: longer secrets
Make a random 12-bit secret, run Bernstein–Vazirani once with **one shot**, and check that you
get the secret back. How many classical questions would the same secret need?

In [ ]:
rng = np.random.default_rng(2026)
long_secret = "".join(rng.choice(["0", "1"], size=12))
# TODO: run bv_circuit(long_secret) with shots=1 and print the single result next to long_secret

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
rng = np.random.default_rng(2026)
long_secret = "".join(rng.choice(["0", "1"], size=12))
result = sampler.run([bv_circuit(long_secret)], shots=1).result()[0].data.c.get_counts()
print("secret  :", long_secret)
print("one shot:", list(result)[0])
print("Classically: 12 questions, one per bit. Quantum: one query and one shot.")